In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [2]:
BASE_OUTPUT_DIR =  "Cross Validation Final Results Beta-VAE"
N_FOLDS = 5

In [3]:
# =========================================================
# METRIC CALCULATION FUNCTION (DUAL THRESHOLD SUPPORT)
# =========================================================
def calculate_fold_metrics(csv_path, fold_num):
    """
    Calculates classification metrics for both Statistical and Grid Search
    thresholds using the test_results.csv.
    """
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f" Could not read file: {e}")
        return []

    # 1 = Fungi (Normal/Positive), 0 = Bacteria (Anomaly/Negative)
    y_true = df['True_Label_Binary']
    raw_scores = df['L1_Loss']        # Raw reconstruction error
    results_list = []

    # Define the two approaches
    approaches = [
        ("Statistical (Mean+2Std)", "Pred_MeanStd", "Threshold_MeanStd"),
        ("Grid Search (F1 Opt)",    "Pred_Grid",    "Threshold_Grid")
    ]

    for name, pred_col, thresh_col in approaches:
        y_pred = df[pred_col]

        threshold_val = df[thresh_col].iloc[0]

        # Confusion Matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        # Metrics
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        # Specificity = Recall for Bacteria (TN / (TN + FP))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        accuracy = (tp + tn) / (tp + tn + fp + fn)

        results_list.append({
            "Fold": fold_num,
            "Method": name,
            "Threshold": float(threshold_val),
            "AUC": (specificity+recall)/2,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "Specificity": specificity,
            "F1_Score": f1
        })

    return results_list

In [5]:
# ============================================================
# AGGREGATE & PRINT RESULTS
# ============================================================

import pandas as pd
import numpy as np
import os


all_results = []
print("Aggregating Results across all folds:")
print("NOTE: Metrics below treat FUNGI as the NEGATIVE class (0) and BACTERIA as POSITIVE (1)")

for fold in range(1, N_FOLDS + 1):
    csv_path = os.path.join(BASE_OUTPUT_DIR, f"fold_{fold}", "test_results.csv")

    if os.path.exists(csv_path):
        fold_metrics = calculate_fold_metrics(csv_path, fold)
        all_results.extend(fold_metrics)

if not all_results:
    print("No results found. Check your file paths.")
else:
    df_summary = pd.DataFrame(all_results)

    def print_formatted_report(df, method_name):
        # Filter for the specific method
        subset = df[df['Method'] == method_name].copy()
        # Calculate stats
        means = subset.mean(numeric_only=True)
        stds = subset.std(numeric_only=True)

        print(f"\n========================================")
        print(f"METHOD: {method_name}")
        print(f"========================================")

        # The metrics to display
        metrics = ["Precision", "Recall", "Specificity", "F1_Score", "AUC"]

        for m in metrics:
            mean_val = means.get(m, 0)
            std_val = stds.get(m, 0)
            print(f"🔹 {m:<12}: {mean_val:.4f} ± {std_val:.4f}")

        # Print the average Threshold used
        thresh_mean = means.get("Threshold", 0)
        thresh_std = stds.get("Threshold", 0)
        print(f"🔹 {'Threshold':<12}: {thresh_mean:.4f} ± {thresh_std:.4f}")

        print("-" * 40)
        print("Full DataFrame (Per Fold):")
        display_cols = ["Fold", "Precision", "Recall", "Specificity", "F1_Score", "AUC", "Threshold"]
        print(subset[display_cols].to_string(index=False))
        print("\n")

    # Print Report for Statistical Threshold (Mean + 2*Std)
    print_formatted_report(df_summary, "Statistical (Mean+2Std)")

    # Print Report for Grid Search Threshold
    print_formatted_report(df_summary, "Grid Search (F1 Opt)")

    # Save the raw combined data
    save_path = os.path.join(BASE_OUTPUT_DIR, "final_comparison_results.csv")
    df_summary.to_csv(save_path, index=False)
    print(f"Full comparison data saved to {save_path}")

Aggregating Results across all folds:
NOTE: Metrics below treat FUNGI as the NEGATIVE class (0) and BACTERIA as POSITIVE (1)

METHOD: Statistical (Mean+2Std)
🔹 Precision   : 0.7636 ± 0.0144
🔹 Recall      : 0.9943 ± 0.0021
🔹 Specificity : 0.9293 ± 0.0057
🔹 F1_Score    : 0.8638 ± 0.0087
🔹 AUC         : 0.9618 ± 0.0024
🔹 Threshold   : 0.1148 ± 0.0005
----------------------------------------
Full DataFrame (Per Fold):
 Fold  Precision   Recall  Specificity  F1_Score      AUC  Threshold
    1   0.764706 0.995215     0.929786  0.864865 0.962501   0.114546
    2   0.748654 0.997608     0.923204  0.855385 0.960406   0.114508
    3   0.768519 0.992823     0.931432  0.866388 0.962127   0.114371
    4   0.751812 0.992823     0.924849  0.855670 0.958836   0.114817
    5   0.784499 0.992823     0.937431  0.876452 0.965127   0.115720



METHOD: Grid Search (F1 Opt)
🔹 Precision   : 0.9672 ± 0.0120
🔹 Recall      : 0.9675 ± 0.0134
🔹 Specificity : 0.9924 ± 0.0029
🔹 F1_Score    : 0.9672 ± 0.0076
🔹 AUC   